# Economic Risk Mapping of Horticulture Under Shifting Climatic Zones
### Exploratory Data Analysis, Feature Engineering & Risk Modeling

- **Region**: Kashmir Valley (*Shopian, Baramulla, Pulwama*)
- **Target Crops**: Apples, Walnuts, Saffron
- **Primary Objective**: Predict per-hectare financial loss and quantify climate vulnerability

## 1. Setup & Environment Imports

In [ ]:
import sys
import os
from pathlib import Path

# Add src to Python path
project_root = Path(os.path.abspath('..' if os.path.exists('../src') else '.'))
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from data_generation import SyntheticDataGenerator
from data_preprocessing import DataPreprocessor, preprocess_data
from model_training import XGBoostTrainer
from model_evaluation import ModelEvaluator, FinancialVulnerabilityCalculator

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
print('Setup complete. All modules successfully imported.')

## 2. Load and Inspect Raw Datasets
We load the three primary data pillars: climate records, geospatial vegetation indicators, and economic harvest metrics.

In [ ]:
climate_path = project_root / 'data/synthetic/climate_data.csv'
geo_path = project_root / 'data/synthetic/geospatial_data.csv'
econ_path = project_root / 'data/synthetic/economic_data.csv'

# If data is not yet generated, create it
if not (climate_path.exists() and geo_path.exists() and econ_path.exists()):
    generator = SyntheticDataGenerator(random_seed=42, n_years=12)
    generator.save_datasets(str(project_root / 'data/synthetic/'))

climate_df = pd.read_csv(climate_path)
geo_df = pd.read_csv(geo_path)
econ_df = pd.read_csv(econ_path)

print(f'Climate: {climate_df.shape[0]:,} records')
print(f'Geospatial: {geo_df.shape[0]:,} records')
print(f'Economic: {econ_df.shape[0]:,} records')
econ_df.head()

## 3. Exploratory Data Analysis (EDA)
### 3.1 Seasonal Temperature Profile Across Districts

In [ ]:
climate_df['Date'] = pd.to_datetime(climate_df['Date'])
climate_df['Month'] = climate_df['Date'].dt.month

monthly_temp = climate_df.groupby(['Month', 'District'])['Avg_Temp'].mean().reset_index()

plt.figure(figsize=(10, 5))
sns.lineplot(data=monthly_temp, x='Month', y='Avg_Temp', hue='District', marker='o', lw=2.5)
plt.title('Monthly Mean Temperature Cycle Across Kashmir Districts', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Average Temperature (deg C)')
plt.xticks(range(1, 13))
plt.show()

### 3.2 Crop Economics: Yield, Pricing, and Revenue per Hectare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=econ_df, x='Crop_Type', y='Revenue_per_Hectare', hue='District', ax=axes[0], palette='Set2')
axes[0].set_title('Revenue per Hectare by Crop & District', fontweight='bold')
axes[0].set_ylabel('Revenue (INR / ha)')

sns.barplot(data=econ_df, x='Crop_Type', y='Yield_per_Hectare', hue='District', ax=axes[1], palette='Set2')
axes[1].set_title('Mean Yield per Hectare (MT / ha)', fontweight='bold')
axes[1].set_ylabel('Yield (MT / ha)')
plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [ ]:
processed_path = project_root / 'data/processed/merged_features.csv'
if not processed_path.exists():
    processed_df = preprocess_data(
        str(climate_path),
        str(geo_path),
        str(econ_path),
        str(processed_path)
    )
else:
    processed_df = pd.read_csv(processed_path)

numeric_cols = processed_df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 8))
corr = processed_df[numeric_cols].corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.5)
plt.title('Correlation Matrix of Climate, Geospatial & Economic Features', fontsize=13, fontweight='bold')
plt.show()

## 5. Model Evaluation & Predictions on Unseen Test Years

In [ ]:
import joblib
model_path = project_root / 'models/xgboost_model.pkl'

if model_path.exists():
    model = joblib.load(model_path)
    trainer = XGBoostTrainer()
    X, y = trainer.prepare_features(processed_df)
    X_train, X_test, y_train, y_test = trainer.temporal_train_test_split(X, y, processed_df, test_years=2)
    y_pred = model.predict(X_test)

    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, y_pred, color='steelblue', alpha=0.7, s=60, label='Predicted Samples')
    diag = np.linspace(min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max()), 100)
    plt.plot(diag, diag, 'r--', label='Perfect Fit')
    plt.xlabel('Actual Revenue per Hectare (INR)')
    plt.ylabel('Predicted Revenue per Hectare (INR)')
    plt.title('Out-of-Time Test Set: Predicted vs Actual Revenue', fontweight='bold')
    plt.legend()
    plt.show()
else:
    print('Model file not found. Run main.py to train the model first.')

## 6. Financial Vulnerability Index & Climate Scenarios (+1.5 deg C & +2.0 deg C)

In [ ]:
v15_path = project_root / 'results/vulnerability_index_1_5C.csv'
v20_path = project_root / 'results/vulnerability_index_2_0C.csv'

if v15_path.exists() and v20_path.exists():
    v15 = pd.read_csv(v15_path)
    v20 = pd.read_csv(v20_path)
    comparison = pd.concat([v15, v20], ignore_index=True)

    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=comparison,
        x='District',
        y='Vulnerability_Index',
        hue='Climate_Scenario',
        palette=['skyblue', 'salmon']
    )
    plt.title('Financial Vulnerability Index by District Under Climate Scenarios', fontsize=13, fontweight='bold')
    plt.ylabel('Vulnerability Index (0-100)')
    plt.ylim(0, 100)
    plt.show()

    print(comparison[['District', 'Crop_Type', 'Climate_Scenario', 'Revenue_Loss_Percentage', 'Vulnerability_Index']])
else:
    print('Vulnerability scenario CSVs not found. Run main.py first.')